# Model Training — Retail Demand Forecasting

Trains **LightGBM** and **RandomForest** regressors on a pandas feature frame
to predict daily product demand per store-product combination.

Both models are logged with **MLflow autologging** and **registered in the
workspace model registry** (`retail_demand_lgbm`, `retail_demand_rf`), so each
appears as an ML model item alongside the experiment. Training uses named,
primitive-typed pandas columns, which gives the models a **column-based
signature** — required to publish a model version as a REST endpoint.

**Target:** `daily_quantity` (continuous — units sold per day)

**Reads from:** `gold_ml_features`

**Writes to:** `gold_ml_model_metrics`, MLflow registry

Same-day leakage columns (`daily_revenue`, `transaction_count`) are excluded —
the model predicts demand from lagged demand, price, discount, calendar and
store/product attributes only.

In [ ]:
import mlflow
import mlflow.lightgbm
import mlflow.sklearn
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, when, isnan

spark = SparkSession.builder.getOrCreate()

EXPERIMENT_NAME = 'retail-demand-forecasting'
mlflow.set_experiment(EXPERIMENT_NAME)
print(f'Spark session ready — MLflow experiment: {EXPERIMENT_NAME}')

In [ ]:
features_df = spark.read.table('gold_ml_features')
print(f'Feature rows: {features_df.count():,}')

# Clean nulls / NaN in numeric columns so the trainer never sees invalid values
for c, dtype in features_df.dtypes:
    if dtype in ('double', 'float'):
        features_df = features_df.withColumn(c, when(col(c).isNull() | isnan(col(c)), lit(0.0)).otherwise(col(c)))
    elif dtype in ('int', 'bigint', 'long'):
        features_df = features_df.withColumn(c, when(col(c).isNull(), lit(0)).otherwise(col(c)))

features_df.select('daily_quantity').describe().show()

In [ ]:
# === Build a pandas feature frame with NAMED, PRIMITIVE-TYPED columns ===
# No VectorAssembler: a single vector column gives the logged model a tensor
# schema, and Fabric cannot publish tensor-schema models as REST endpoints
# ("This model's schema is either empty or has tensors"). Named float columns
# produce a column-based signature, which endpoints require.

# Numeric predictors — EXCLUDE same-day leakage (daily_revenue, transaction_count)
numeric_features = [
    'avg_discount', 'avg_price',
    'day_of_week', 'day_of_month', 'month', 'week_of_year', 'is_weekend',
    'demand_lag_1', 'demand_lag_7', 'revenue_lag_1',
    'unit_cost', 'margin_pct',
]
cat_cols = ['category', 'subcategory', 'region', 'store_format']
LABEL = 'daily_quantity'

pdf = features_df.select(numeric_features + cat_cols + [LABEL]).toPandas()

# Deterministic ordinal encoding for categoricals (saved for scoring reuse)
category_maps = {}
for c in cat_cols:
    cats = sorted(pdf[c].astype(str).fillna('unknown').unique())
    category_maps[c] = {v: i for i, v in enumerate(cats)}
    pdf[f'{c}_idx'] = pdf[c].astype(str).map(category_maps[c]).fillna(-1).astype('float64')

feature_cols = numeric_features + [f'{c}_idx' for c in cat_cols]

# Endpoint-safe dtypes: every feature a named float64 column, no vectors/tensors
X = pdf[feature_cols].astype('float64')
y = pdf[LABEL].astype('float64')

print(f'Model-ready rows: {len(X):,}')
print(f'Feature count: {len(feature_cols)}')
X.dtypes.value_counts()

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train):,} rows')
print(f'Test:  {len(X_val):,} rows')

In [ ]:
# lgbm_model
# Autolog captures params/metrics/plots, but we log the model EXPLICITLY:
# autolog's inferred signature has a tensor (numpy) output, which Fabric
# endpoints reject ("schema is either empty or has tensors"). An explicit
# infer_signature with a named pandas output column keeps both sides
# column-based, so the model version can be published as a REST endpoint.
from mlflow.models.signature import infer_signature

mlflow.lightgbm.autolog(log_models=False)
mlflow.sklearn.autolog(log_models=False)

lgbm_model = LGBMRegressor(
    learning_rate=0.07,
    max_delta_step=2,
    n_estimators=100,
    max_depth=10,
    objective='regression',
    random_state=42,
)

with mlflow.start_run(run_name='lgbm_demand') as run:
    lgbm_run_id = run.info.run_id  # Capture run_id for model prediction later
    lgbm_model.fit(X_train, y_train)  # named float64 columns -> column-based signature
    y_pred_lgbm = lgbm_model.predict(X_val)
    rmse_lgbm = float(np.sqrt(mean_squared_error(y_val, y_pred_lgbm)))
    mae_lgbm = float(mean_absolute_error(y_val, y_pred_lgbm))
    r2_lgbm = float(r2_score(y_val, y_pred_lgbm))

    signature = infer_signature(X_val, pd.DataFrame({'predicted_demand': y_pred_lgbm}))
    mlflow.lightgbm.log_model(
        lgbm_model,
        'model',
        signature=signature,
        input_example=X_val.head(5),
        registered_model_name='retail_demand_lgbm',  # Register the trained model
    )

print(f'LightGBM run: {lgbm_run_id}')
print(f'  RMSE {rmse_lgbm:.4f} | MAE {mae_lgbm:.4f} | R2 {r2_lgbm:.4f}')

In [ ]:
# rf_model — second registered model, mirroring the multi-model workspace layout
rf_model = RandomForestRegressor(
    n_estimators=80,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
)

with mlflow.start_run(run_name='rf_demand') as run:
    rf_run_id = run.info.run_id
    rf_model.fit(X_train, y_train)
    y_pred_rf = rf_model.predict(X_val)
    rmse_rf = float(np.sqrt(mean_squared_error(y_val, y_pred_rf)))
    mae_rf = float(mean_absolute_error(y_val, y_pred_rf))
    r2_rf = float(r2_score(y_val, y_pred_rf))

    signature = infer_signature(X_val, pd.DataFrame({'predicted_demand': y_pred_rf}))
    mlflow.sklearn.log_model(
        rf_model,
        'model',
        signature=signature,
        input_example=X_val.head(5),
        registered_model_name='retail_demand_rf',
    )

print(f'RandomForest run: {rf_run_id}')
print(f'  RMSE {rmse_rf:.4f} | MAE {mae_rf:.4f} | R2 {r2_rf:.4f}')

# Champion = better validation RMSE; scoring loads this from the registry
champion_name, champion_run, rmse, mae, r2 = (
    ('retail_demand_lgbm', lgbm_run_id, rmse_lgbm, mae_lgbm, r2_lgbm)
    if rmse_lgbm <= rmse_rf
    else ('retail_demand_rf', rf_run_id, rmse_rf, mae_rf, r2_rf)
)
print(f'\n=== Model Evaluation ===')
print(f'Champion: {champion_name}')
print(f'RMSE: {rmse:.4f}')
print(f'MAE:  {mae:.4f}')
print(f'R²:   {r2:.4f}')

In [ ]:
metrics_df = spark.createDataFrame(
    [('retail-sales', 'demand-forecasting', champion_name,
      len(feature_cols), len(X_train), len(X_val),
      float(rmse), float(mae), float(r2))],
    ['demo_id', 'use_case', 'model_type', 'feature_count',
     'train_rows', 'test_rows', 'rmse', 'mae', 'r2']
).withColumn('trained_at', current_timestamp())

metrics_df.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_model_metrics')
print('Model metrics saved to gold_ml_model_metrics')
metrics_df.show(truncate=False)

In [ ]:
# Persist scoring metadata: champion model name, feature order, category maps.
# Downstream notebooks load the model from the MLflow REGISTRY (not Files/).
import json

scoring_meta = {
    'champion_model': champion_name,
    'champion_run_id': champion_run,
    'feature_cols': feature_cols,
    'numeric_features': numeric_features,
    'cat_cols': cat_cols,
    'category_maps': category_maps,
}
meta_df = spark.createDataFrame([(json.dumps(scoring_meta),)], ['meta_json'])
meta_df.write.mode('overwrite').format('delta').saveAsTable('gold_ml_scoring_meta')

print(f"Registered models: retail_demand_lgbm, retail_demand_rf")
print(f"Champion for scoring: {champion_name}")
print('Scoring metadata saved to gold_ml_scoring_meta')